# Fine-Tuning LLMs with LoRA (Low-Rank Adaptation)

Fine-tuning is how you take a general-purpose LLM and make it **your** model — specialized for your domain, your data, your task.

## Why Fine-Tune?

| Approach | When to Use | Cost |
| :--- | :--- | :--- |
| **Prompt Engineering** | Simple tasks, general knowledge | Free (just text) |
| **RAG** | Need access to specific documents | Low (embeddings + DB) |
| **Fine-tuning** | Need specialized behavior, style, or domain expertise | Medium (GPU time) |
| **Full Training** | Building a new model from scratch | Very high ($M+) |

### Real-World Fine-Tuning Examples
- A hospital fine-tunes Llama to understand medical radiology reports
- A law firm fine-tunes Mistral to draft legal documents in their style
- A GIS company fine-tunes to generate QGIS Python scripts from natural language

## The Problem: Full Fine-tuning is Expensive

Fine-tuning ALL parameters of an 8B model requires:
- ~32GB+ VRAM (GPU memory)
- Hours of training time
- The full model in float32 = ~32GB on disk

Most of us don't have that hardware. Enter **LoRA**.

## 1. What is LoRA?

**LoRA (Low-Rank Adaptation)** is a clever trick: instead of updating ALL the model's parameters, we **freeze** the original model and add tiny "adapter" layers that get trained.

### The Math (Simplified)

A neural network layer does: `output = W × input`  (where W is a huge matrix, e.g., 4096×4096)

LoRA says: Instead of updating W directly, add a small correction:
```
output = W × input + (B × A) × input
```
Where:
- **W** stays frozen (4096×4096 = 16.7M parameters — NOT trained)
- **A** is tiny (4096×8 = 32K parameters — trained)
- **B** is tiny (8×4096 = 32K parameters — trained)

So instead of training 16.7M parameters per layer, we train only 64K. That's a **99.6% reduction!**

### QLoRA: Even More Efficient
**QLoRA** combines LoRA with quantization — the frozen model is loaded in 4-bit (taking ~4GB instead of ~16GB), and the tiny LoRA adapters train in float16. This lets you fine-tune an 8B model on a **single consumer GPU with 8GB VRAM**.

```
Full Fine-tuning:  Entire model trainable   → 32GB+ VRAM needed
LoRA:              Only adapters trainable   → 16GB VRAM needed
QLoRA:             Quantized model + LoRA    → 8GB VRAM needed ✅
```

## 2. Setup: Install Required Libraries

In [1]:
!pip install transformers peft datasets accelerate bitsandbytes trl torch -q

### Library Roles

| Library | What it Does |
| :--- | :--- |
| `transformers` | Core model loading and inference |
| `peft` | Parameter-Efficient Fine-Tuning (LoRA, QLoRA) |
| `datasets` | Load and process training datasets |
| `trl` | Training helpers (SFTTrainer for instruction tuning) |
| `bitsandbytes` | 4-bit/8-bit quantization (for QLoRA) |
| `accelerate` | Multi-GPU and hardware optimization |

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected. Fine-tuning will be slower on CPU.")
    print("   This notebook will still work for learning, but real fine-tuning benefits greatly from a GPU.")

PyTorch version: 2.10.0+cpu
CUDA available: False
⚠️ No GPU detected. Fine-tuning will be slower on CPU.
   This notebook will still work for learning, but real fine-tuning benefits greatly from a GPU.


## 3. Load a Small Base Model

We'll use **GPT-2** (124M params) to keep things fast and runnable on any hardware. The concepts transfer directly to larger models like Llama 3.

In [3]:
model_name = "gpt2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a pad token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Use float16 if you have a GPU
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.0f} MB (float32)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model: gpt2
Total parameters: 124,439,808
Model size: ~475 MB (float32)


### Before Fine-Tuning: Test the Base Model

Let's see what GPT-2 outputs on our target domain BEFORE training. This gives us a baseline.

In [4]:
def generate_text(model, tokenizer, prompt, max_tokens=60):
    """Generate text from a prompt using the given model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, 
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test BASE model
test_prompt = "Explain NDVI in remote sensing:"
print("BEFORE fine-tuning:")
print(generate_text(model, tokenizer, test_prompt))
print("\n" + "="*60)
print("(Likely gibberish or generic text — GPT-2 doesn't know much about remote sensing)")

BEFORE fine-tuning:
Explain NDVI in remote sensing: Using NDVI to test for NDVI is not a new technology. However, many companies are already using it to detect low-frequency (LE) signals and to identify high-frequency (HFP) signals. The first attempt was made to detect NDVI using the NMR technique, but it

(Likely gibberish or generic text — GPT-2 doesn't know much about remote sensing)


## 4. Prepare Training Data

For fine-tuning, you need examples in the domain you want the model to learn. The format depends on your goal:

- **Instruction tuning**: `"Question: ... Answer: ..."` pairs
- **Style transfer**: Text in the style you want
- **Domain adaptation**: Raw text from your domain

Let's create a small geospatial Q&A dataset:

In [5]:
# Our geospatial training data (in practice, you'd have hundreds or thousands of examples)
training_data = [
    "Q: What is NDVI? A: NDVI (Normalized Difference Vegetation Index) measures vegetation health using the formula (NIR - Red) / (NIR + Red). Values range from -1 to 1, where higher values indicate healthy vegetation.",
    "Q: What is SAR imagery? A: SAR (Synthetic Aperture Radar) is an active remote sensing technology that emits microwave pulses and measures their reflection. Unlike optical imagery, SAR works through clouds and at night.",
    "Q: What is a GeoTIFF? A: A GeoTIFF is a standard image format that includes geographic metadata (coordinate system, projection, extent) embedded in the file. It's the most common format for raster geospatial data.",
    "Q: What is Sentinel-2? A: Sentinel-2 is an ESA satellite constellation providing multispectral optical imagery at 10m resolution with a 5-day revisit time. It has 13 spectral bands covering visible, NIR, and SWIR wavelengths.",
    "Q: What is land cover classification? A: Land cover classification is the process of categorizing satellite imagery pixels into classes like urban, forest, water, and agriculture using machine learning or deep learning algorithms.",
    "Q: What is QGIS? A: QGIS is a free, open-source Geographic Information System for creating, editing, visualizing, and analyzing geospatial data. It supports vector and raster formats and has a rich plugin ecosystem.",
    "Q: What is a spectral index? A: A spectral index is a mathematical combination of satellite bands that highlights specific features. Examples include NDVI for vegetation, NDWI for water, and NDBI for built-up areas.",
    "Q: What is Google Earth Engine? A: Google Earth Engine (GEE) is a cloud platform for planetary-scale geospatial analysis. It provides petabytes of satellite imagery and a JavaScript/Python API for processing without downloading data.",
    "Q: What is spatial resolution? A: Spatial resolution is the size of the smallest feature a sensor can detect. Sentinel-2 has 10m resolution (each pixel = 10m x 10m on the ground), while WorldView has 30cm resolution.",
    "Q: What is a coordinate reference system? A: A CRS defines how geographic locations map to a flat surface. Common CRSs include WGS 84 (EPSG:4326) for latitude/longitude and UTM zones for projected coordinates.",
]

# Convert to HuggingFace Dataset
dataset = Dataset.from_dict({"text": training_data})
print(f"Training examples: {len(dataset)}")
print(f"\nSample:")
print(dataset[0]['text'])

Training examples: 10

Sample:
Q: What is NDVI? A: NDVI (Normalized Difference Vegetation Index) measures vegetation health using the formula (NIR - Red) / (NIR + Red). Values range from -1 to 1, where higher values indicate healthy vegetation.


## 5. Configure LoRA

This is where we define HOW the LoRA adapters work. The key parameters:

- **r (rank)**: Size of the adapter matrices. Higher = more capacity but more memory. 8-64 is typical.
- **lora_alpha**: Scaling factor. Usually set to 2× the rank.
- **target_modules**: Which layers to add adapters to. Attention layers are the most effective.
- **lora_dropout**: Regularization to prevent overfitting.

In [6]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # We're doing text generation
    r=8,                           # Rank — small = fewer params, large = more capacity
    lora_alpha=16,                 # Scaling factor (usually 2x rank)
    lora_dropout=0.1,              # Dropout for regularization
    target_modules=["c_attn"],     # GPT-2's attention layer name
)

# Apply LoRA to the model
peft_model = get_peft_model(model, lora_config)

# Let's see the parameter efficiency!
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())

print(f"Total parameters:     {total:>12,}")
print(f"Trainable parameters: {trainable:>12,}")
print(f"Trainable %:          {100 * trainable / total:>11.2f}%")
print(f"\n→ We're only training {trainable/total*100:.2f}% of the model!")

Total parameters:      124,734,720
Trainable parameters:      294,912
Trainable %:                 0.24%

→ We're only training 0.24% of the model!


d:\setups\ml-dl\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## 6. Tokenize the Training Data

In [7]:
def tokenize_function(examples):
    """Tokenize the text data for training."""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokenized dataset: {tokenized_dataset}")
print(f"Sample token length: {len(tokenized_dataset[0]['input_ids'])}")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenized dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})
Sample token length: 128


## 7. Train!

Now we actually fine-tune. With only 10 examples and a small model, this should take less than a minute even on CPU.

In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=15,          # More epochs for our tiny dataset
    per_device_train_batch_size=2,
    learning_rate=3e-4,           # LoRA benefits from higher LR than full fine-tuning
    logging_steps=5,
    save_strategy="no",           # Don't save checkpoints for this demo
    report_to="none",             # Don't report to wandb/tensorboard
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("\n✅ Training complete!")

Starting LoRA fine-tuning...


d:\setups\ml-dl\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.756513
10,3.661102
15,3.639632
20,3.501928
25,3.416277
30,3.335196
35,3.346177
40,3.291070
45,3.212851
50,3.199673



✅ Training complete!


## 8. After Fine-Tuning: Test the Results

Let's see if the model learned anything from our geospatial Q&A data.

In [9]:
# Test on a question from our training data
print("Test 1 — Seen during training:")
print(generate_text(peft_model, tokenizer, "Q: What is NDVI? A:"))
print("\n" + "="*60)

# Test on a SIMILAR question (not in training data)
print("\nTest 2 — Unseen, but related:")
print(generate_text(peft_model, tokenizer, "Q: What is NDWI? A:"))
print("\n" + "="*60)

# Test on a general prompt
print("\nTest 3 — General prompt:")
print(generate_text(peft_model, tokenizer, "Explain NDVI in remote sensing:"))

Test 1 — Seen during training:
Q: What is NDVI? A: NDVI is a 3D graphics projection system for Windows that allows developers to create 3D graphics for their own applications. It is a standard for rendering software that allows developers to create 3D graphics that are projected to the same size as the real world. It is not a standard for rendering software,


Test 2 — Unseen, but related:
Q: What is NDWI? A: NDWI is an Open Source (OOP) technology that enables the production of wireless data. NDWI is an Open Source (OOP) technology that enables the production of wireless data. A: NDWI is a free open source (OOP) technology. B: NDWI is a free


Test 3 — General prompt:
Explain NDVI in remote sensing: The Remote Sensing Solution (RSIS) project is a research and development effort that seeks to enable remote sensing technologies to be used in real-time to improve the performance of remote sensing systems. The project focuses on developing a system that can be used to detect the presence 

**What to expect:**
- Test 1 should show improved output (the model memorized this)
- Test 2 is the real test — can it generalize to related topics?
- With only 10 examples, generalization will be limited. In production, you'd use 1,000-10,000+ examples.

**Important:** GPT-2 is a very old, small model. The results won't be impressive. But the *process* is identical to fine-tuning Llama 3, Mistral, or any modern model — just swap the model name and scale up the data.

## 9. Saving and Loading LoRA Adapters

One of the best things about LoRA: the adapter weights are **tiny** (often < 10MB). You can save and share them easily.

In [10]:
# Save the LoRA adapters (NOT the full model — just the tiny adapter weights)
adapter_path = "./geo_lora_adapter"
peft_model.save_pretrained(adapter_path)

import os
adapter_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"Adapter saved to: {adapter_path}")
print(f"Adapter size: {adapter_size / 1024:.1f} KB")
print(f"Full model size: ~{total_params * 4 / 1024**2:.0f} MB")
print(f"\n→ LoRA adapter is {total_params * 4 / adapter_size:.0f}x smaller than the full model!")

Adapter saved to: ./geo_lora_adapter
Adapter size: 1161.0 KB
Full model size: ~475 MB

→ LoRA adapter is 419x smaller than the full model!


In [11]:
# To load the adapter later:
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("gpt2")

# Apply saved LoRA adapter
loaded_model = PeftModel.from_pretrained(base_model, adapter_path)

# Test the loaded model
print("Loaded model output:")
print(generate_text(loaded_model, tokenizer, "Q: What is SAR imagery? A:"))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded model output:
Q: What is SAR imagery? A: SAR is a type of image produced by a camera or computer to capture and store data about a person's location. SAR images are typically based on images taken by satellites, ground stations, or other sources. SAR images are typically based on the most recent satellite images taken by a computer or satellite. They


## 10. Scaling Up: What Changes for Real Fine-Tuning

To go from this demo to production fine-tuning:

| Aspect | This Demo | Production |
| :--- | :--- | :--- |
| **Model** | GPT-2 (124M) | Llama 3 (8B), Mistral (7B) |
| **Data** | 10 examples | 1,000-100,000+ examples |
| **Technique** | LoRA (FP32) | QLoRA (4-bit base + FP16 adapters) |
| **GPU** | Optional | Required (8GB+ VRAM) |
| **Training time** | Seconds | Hours-days |
| **LoRA rank** | 8 | 16-64 |

### Useful Datasets for Geospatial Fine-Tuning
- [BigEarthNet](https://huggingface.co/datasets/imvladikon/bigearth_net) — Satellite imagery captions
- Custom Q&A from your GIS documentation
- Generated pairs from GPT-4 about remote sensing topics

---

**Key takeaway:** LoRA makes fine-tuning accessible. You don't need a data center — a laptop with a decent GPU can fine-tune a 7B model in a few hours.